# Proyecto II Parcial - Modelado Avanzado de Base de Datos

## Notebook 01: Extracción y auditoría de archivos Parquet

Este notebook realiza la **Fase 1: Extracción de datos**.

Objetivo principal:

- Revisar que los archivos Parquet estén en la capa `raw`.
- Leer cada archivo de forma individual con Spark.
- Detectar archivos válidos, vacíos, ilegibles o corruptos.
- Registrar errores sin detener todo el proceso.
- Crear la tabla `audit_file_inventory`.
- Crear evidencia de archivos enviados a `quarantine`.



### Alcance de las clasificaciones de error

En esta fase, `read_status` responde una pregunta operativa: **¿Spark pudo leer y contar el archivo?** Por eso solo usa `READ_OK` o `READ_ERROR` dentro de `audit_file_inventory`.

`error_category`, en cambio, pertenece a la clasificación fina de calidad definida para la Fase 3 y contempla siete categorías de diagnóstico. En este notebook solo se conserva una referencia preliminar para cuarentena; la asignación y justificación detallada de esas categorías se profundizará en el notebook `02_diagnostico_reconstruccion.ipynb`.

## 1. Explicación rápida de la arquitectura

El proyecto usa una estructura tipo **Data Lakehouse**:

| Capa | Para qué sirve |
|---|---|
| `raw` | Guarda los archivos originales sin modificar. |
| `bronze` | Guarda datos leídos y reconstruidos. |
| `silver` | Guarda datos limpios y validados. |
| `gold` | Guarda tablas finales para análisis. |
| `quarantine` | Guarda referencias de archivos o registros rechazados. |
| `audit` | Guarda métricas e inventarios del procesamiento. |

En este notebook trabajamos principalmente con:

- `data/raw`
- `data/quarantine`
- `data/audit`

In [1]:
# Librerías principales
# Estas librerías se usan para manejar rutas, fechas, hashes, errores y Spark.

import os
import sys
import re
import json
import uuid
import hashlib
import traceback
from pathlib import Path
from datetime import datetime

try:
    import yaml
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Falta instalar PyYAML en el Python/kernel que ejecuta este notebook. "
        f"Ejecuta: {sys.executable} -m pip install PyYAML"
    ) from exc

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType,
    LongType, TimestampType
)

import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print("HADOOP_HOME =", os.environ.get("HADOOP_HOME"))

print("Librerías cargadas correctamente")

HADOOP_HOME = C:\hadoop
Librerías cargadas correctamente


In [2]:
# Configuración del proyecto
# Esta celda detecta automáticamente la carpeta raíz del proyecto.

CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "No se encontró la carpeta data. "
        "Abre Jupyter desde etl_spark_parquet_advanced o desde la carpeta notebooks."
    )

DATA_PATH = PROJECT_ROOT / "data"
RAW_PATH = DATA_PATH / "raw"
BRONZE_PATH = DATA_PATH / "bronze"
SILVER_PATH = DATA_PATH / "silver"
GOLD_PATH = DATA_PATH / "gold"
QUARANTINE_PATH = DATA_PATH / "quarantine"
AUDIT_PATH = DATA_PATH / "audit"
CONFIG_PATH = PROJECT_ROOT / "config"
METADATA_PATH = PROJECT_ROOT / "metadata"

# Crear carpetas si faltan
for path in [RAW_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUARANTINE_PATH, AUDIT_PATH, CONFIG_PATH, METADATA_PATH]:
    path.mkdir(parents=True, exist_ok=True)

# process_id único para esta ejecución
# Sirve para saber qué datos pertenecen a esta corrida del pipeline.
PROCESS_ID = str(uuid.uuid4())
PROCESS_START_TIME = datetime.now()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_PATH:", RAW_PATH)
print("AUDIT_PATH:", AUDIT_PATH)
print("QUARANTINE_PATH:", QUARANTINE_PATH)
print("PROCESS_ID:", PROCESS_ID)

PROJECT_ROOT: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced
RAW_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw
AUDIT_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\audit
QUARANTINE_PATH: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\quarantine
PROCESS_ID: 2e9a7366-d209-4985-ad78-8c8ca2a14476


In [3]:
# Archivo de configuración
# Permite que el pipeline acepte parámetros externos mediante un archivo de configuración.
# Si el archivo no existe, aquí se crea uno.

config_file = CONFIG_PATH / "etl_config.yaml"

default_config = {
    "project_name": "etl_spark_parquet_advanced",
    "source_system_nyc": "NYC_TLC",
    "source_system_bad": "APACHE_PARQUET_TESTING",
    "raw_path": str(RAW_PATH),
    "audit_path": str(AUDIT_PATH),
    "quarantine_path": str(QUARANTINE_PATH),
    "bronze_path": str(BRONZE_PATH),
    "read_mode": "individual_files",
    "database_engine": "duckdb",
    "process_all_files": True
}

if not config_file.exists():
    with open(config_file, "w", encoding="utf-8") as f:
        yaml.dump(default_config, f, allow_unicode=True, sort_keys=False)
    print("Archivo de configuración creado:", config_file)
else:
    print("Archivo de configuración ya existe:", config_file)

with open(config_file, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

config

Archivo de configuración ya existe: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\config\etl_config.yaml


{'project_name': 'etl_spark_parquet_advanced',
 'source_system_nyc': 'NYC_TLC',
 'source_system_bad': 'APACHE_PARQUET_TESTING',
 'raw_path': 'c:\\Users\\Usuario\\Downloads\\etl_spark_parquet_advanced\\etl_spark_parquet_advanced\\data\\raw',
 'audit_path': 'c:\\Users\\Usuario\\Downloads\\etl_spark_parquet_advanced\\etl_spark_parquet_advanced\\data\\audit',
 'quarantine_path': 'c:\\Users\\Usuario\\Downloads\\etl_spark_parquet_advanced\\etl_spark_parquet_advanced\\data\\quarantine',
 'bronze_path': 'c:\\Users\\Usuario\\Downloads\\etl_spark_parquet_advanced\\etl_spark_parquet_advanced\\data\\bronze',
 'silver_path': 'c:\\Users\\Usuario\\Downloads\\etl_spark_parquet_advanced\\etl_spark_parquet_advanced\\data\\silver',
 'gold_path': 'c:\\Users\\Usuario\\Downloads\\etl_spark_parquet_advanced\\etl_spark_parquet_advanced\\data\\gold',
 'metadata_path': 'c:\\Users\\Usuario\\Downloads\\etl_spark_parquet_advanced\\etl_spark_parquet_advanced\\metadata',
 'database_path': 'c:\\Users\\Usuario\\Downlo

In [4]:
# Iniciar Spark
# Esta configuración indica a Spark qué Python debe usar en Windows.

import os
import sys
from pyspark.sql import SparkSession

PYTHON_PATH = sys.executable

os.environ["PYSPARK_PYTHON"] = PYTHON_PATH
os.environ["PYSPARK_DRIVER_PYTHON"] = PYTHON_PATH

spark = (
    SparkSession.builder
    .appName("NYC_TAXI_ETL_01_EXTRACCION")
    .master("local[*]")
    .config("spark.pyspark.python", PYTHON_PATH)
    .config("spark.pyspark.driver.python", PYTHON_PATH)
    .config("spark.sql.session.timeZone", "America/Guayaquil")
    .config("spark.hadoop.io.native.lib.available", "false")
    .config("spark.sql.files.ignoreCorruptFiles", "false")
    .config("spark.sql.files.ignoreMissingFiles", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark iniciado correctamente")
print("Versión de Spark:", spark.version)
print("Python usado por Spark:", PYTHON_PATH)

Spark iniciado correctamente
Versión de Spark: 4.1.2
Python usado por Spark: c:\Users\Usuario\AppData\Local\Programs\Python\Python311\python.exe


In [5]:
# Función auxiliar para que Spark lea rutas de Windows sin problema.
# Convierte C:\carpeta\archivo.parquet en C:/carpeta/archivo.parquet.

def spark_path(path):
    return str(Path(path).resolve()).replace("\\", "/")

print("Ejemplo de ruta Spark:")
print(spark_path(RAW_PATH))

Ejemplo de ruta Spark:
C:/Users/Usuario/Downloads/etl_spark_parquet_advanced/etl_spark_parquet_advanced/data/raw


## 2. Verificación de carpetas y archivos

Antes de leer datos, revisamos que la estructura exista.

La capa `raw` conserva los archivos originales sin modificación.

In [6]:
# Verificación rápida de carpetas obligatorias

required_folders = [
    DATA_PATH,
    RAW_PATH,
    RAW_PATH / "yellow",
    RAW_PATH / "green",
    RAW_PATH / "fhvhv",
    RAW_PATH / "bad_parquet",
    BRONZE_PATH,
    SILVER_PATH,
    GOLD_PATH,
    QUARANTINE_PATH,
    AUDIT_PATH,
    CONFIG_PATH,
    METADATA_PATH
]

folder_check = []

for folder in required_folders:
    folder_check.append({
        "folder": str(folder),
        "exists": folder.exists()
    })

folder_check_df = spark.createDataFrame(folder_check)
folder_check_df.show(truncate=False)

+------+-----------------------------------------------------------------------------------------------------+
|exists|folder                                                                                               |
+------+-----------------------------------------------------------------------------------------------------+
|true  |c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data                |
|true  |c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw            |
|true  |c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw\yellow     |
|true  |c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw\green      |
|true  |c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw\fhvhv      |
|true  |c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw\bad_parquet|
|

In [7]:
# Se verifica que existan los archivos mínimos solicitados.

expected_files = [
    "yellow_tripdata_2023-01.parquet",
    "yellow_tripdata_2023-02.parquet",
    "yellow_tripdata_2023-03.parquet",
    "yellow_tripdata_2023-04.parquet",
    "green_tripdata_2023-01.parquet",
    "green_tripdata_2023-02.parquet",
    "fhvhv_tripdata_2023-01.parquet",
    "yellow_tripdata_2022-01.parquet",
    "yellow_tripdata_2022-02.parquet",
    "yellow_tripdata_2021-01.parquet",
    "yellow_tripdata_2020-01.parquet"
]

all_parquet_files = sorted(RAW_PATH.rglob("*.parquet"))
existing_file_names = {p.name for p in all_parquet_files}
missing_files = [f for f in expected_files if f not in existing_file_names]

print("Total de archivos Parquet encontrados en raw:", len(all_parquet_files))

if missing_files:
    print("Archivos mínimos faltantes:")
    for f in missing_files:
        print("-", f)
else:
    print("Todos los archivos mínimos fueron encontrados.")

Total de archivos Parquet encontrados en raw: 16
Todos los archivos mínimos fueron encontrados.


In [8]:
# Verifica los casos de prueba incorporados desde apache/parquet-testing/bad_data.

apache_bad_files = [
    "ARROW-GH-41317.parquet",
    "ARROW-GH-41321.parquet",
    "ARROW-GH-45185.parquet",
    "ARROW-RS-GH-6229-DICTHEADER.parquet",
    "PARQUET-1481.parquet"
]

bad_parquet_path = RAW_PATH / "bad_parquet"
present_bad_files = {p.name for p in bad_parquet_path.glob("*.parquet")}
missing_bad_files = [f for f in apache_bad_files if f not in present_bad_files]

print("Archivos Apache esperados:", len(apache_bad_files))
print("Archivos Apache presentes:", len(apache_bad_files) - len(missing_bad_files))

if missing_bad_files:
    print("Faltan los siguientes archivos; descargar desde el repositorio oficial:")
    for file_name in missing_bad_files:
        download_url = f"https://raw.githubusercontent.com/apache/parquet-testing/master/bad_data/{file_name}"
        print(f"- {file_name}: {download_url}")
else:
    print("Los cinco casos de apache/parquet-testing están disponibles en data/raw/bad_parquet.")

Archivos Apache esperados: 5
Archivos Apache presentes: 5
Los cinco casos de apache/parquet-testing están disponibles en data/raw/bad_parquet.


### Casos incorporados desde `apache/parquet-testing/bad_data`

La descripción del daño se basa en el [README oficial de `bad_data`](https://github.com/apache/parquet-testing/blob/master/bad_data/README.md) y se contrasta con el resultado real capturado por Spark en este notebook:

| Archivo | Caso de prueba y evidencia observada |
|---|---|
| `ARROW-GH-41317.parquet` | Las columnas no tienen el mismo tamaño. Spark lo clasifica `READ_ERROR` y reporta además el tipo lógico no soportado `INT32 (TIME(MILLIS,true))`. |
| `ARROW-GH-41321.parquet` | Los niveles de repetición/definición decodificados son menores que `num_values` del encabezado de página. Spark lo clasifica `READ_ERROR` con el mismo error de tipo lógico no soportado. |
| `ARROW-GH-45185.parquet` | Los niveles de repetición empiezan en 1 en lugar de 0. Spark logra leerlo (`READ_OK`: 5 registros, 1 columna), evidencia de que un caso de `bad_data` no necesariamente falla en todos los lectores. |
| `ARROW-RS-GH-6229-DICTHEADER.parquet` | El encabezado de la página de diccionario declara una cantidad negativa de valores. Spark logra abrirlo (`READ_OK`: 0 registros, 4 columnas), por lo que el defecto no se manifiesta como error de lectura en esta ejecución. |
| `PARQUET-1481.parquet` | Un valor Thrift del esquema está corrupto. Spark falla durante la lectura del footer/metadata porque falta el campo obligatorio `type` en `ColumnMetaData`; queda como `READ_ERROR`. |

Ninguno de los cinco archivos presentes mide 0 bytes. Por tanto, en este conjunto no hay un caso de archivo físicamente vacío; los daños corresponden a metadata, esquema o estructuras internas de página.

## 3. Inventario inicial de archivos

Aquí se crea un listado de todos los `.parquet` encontrados en `data/raw`.

Todavía no se transforma datos. Solo se verifica:

- Qué archivos existen.
- Dónde están.
- A qué servicio pertenecen.
- A qué año y mes pertenecen.
- Cuánto pesan.

In [9]:
# Funciones para extraer metadatos desde la ruta del archivo.

def detect_service_type(file_path):
    parts = [part.lower() for part in Path(file_path).parts]

    if "yellow" in parts:
        return "yellow"
    if "green" in parts:
        return "green"
    if "fhvhv" in parts:
        return "fhvhv"
    if "bad_parquet" in parts:
        return "bad_parquet"

    return "unknown"


def detect_source_system(service_type):
    if service_type == "bad_parquet":
        return config.get("source_system_bad", "APACHE_PARQUET_TESTING")
    return config.get("source_system_nyc", "NYC_TLC")


def extract_partition_value(file_path, prefix):
    # Busca partes como year=2023 o month=01.
    for part in Path(file_path).parts:
        if part.lower().startswith(prefix.lower() + "="):
            value = part.split("=", 1)[1]
            try:
                return int(value)
            except ValueError:
                return None
    return None


def file_size_mb(file_path):
    return round(Path(file_path).stat().st_size / (1024 * 1024), 4)


def clean_error_message(error):
    # Recorta el error para que la tabla sea legible.
    if error is None:
        return None
    text = str(error).replace("\n", " ").replace("\r", " ")
    text = re.sub(r"\s+", " ", text)
    return text[:1000]


def schema_hash(schema):
    # Convierte el esquema en texto y calcula un hash.
    # Si dos archivos tienen el mismo hash, probablemente tienen el mismo esquema.
    schema_text = schema.json()
    return hashlib.md5(schema_text.encode("utf-8")).hexdigest()


def classify_file_error(file_path, error_message):
    # Clasificación sencilla para archivos problemáticos.
    # Esto ayuda a explicar si el archivo es vacío, corrupto o con formato no soportado.

    path = Path(file_path)

    if path.exists() and path.stat().st_size == 0:
        return "NOT_RECOVERABLE_EMPTY_FILE"

    if error_message is None:
        return None

    error_text = str(error_message).lower()

    if "not a parquet" in error_text or "unsupported" in error_text:
        return "NOT_RECOVERABLE_UNSUPPORTED_FORMAT"

    if "corrupt" in error_text or "magic" in error_text or "footer" in error_text or "metadata" in error_text:
        return "NOT_RECOVERABLE_CORRUPT_METADATA"

    if "cannot be cast" in error_text or "parquet column cannot be converted" in error_text:
        return "RECUPERABLE_TYPE_CASTING"

    return "PARTIALLY_RECOVERABLE"

In [10]:
# Crear manifest de archivos encontrados en raw.
# Este manifest es una vista previa antes de leer los datos con Spark.

manifest_rows = []

for file_path in all_parquet_files:
    service_type = detect_service_type(file_path)

    manifest_rows.append({
        "source_system": detect_source_system(service_type),
        "service_type": service_type,
        "file_name": file_path.name,
        "file_path": spark_path(file_path),
        "file_size_mb": file_size_mb(file_path),
        "partition_year": extract_partition_value(file_path, "year"),
        "partition_month": extract_partition_value(file_path, "month")
    })

manifest_df = spark.createDataFrame(manifest_rows)

manifest_df.orderBy("service_type", "partition_year", "partition_month", "file_name").show(100, truncate=False)

+-----------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------------+--------------+------------+----------------------+
|file_name                          |file_path                                                                                                                                          |file_size_mb|partition_month|partition_year|service_type|source_system         |
+-----------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------------+--------------+------------+----------------------+
|ARROW-GH-41317.parquet             |C:/Users/Usuario/Downloads/etl_spark_parquet_advanced/etl_spark_parquet_advanced/data/raw/bad_parquet/ARROW-GH-41317.parquet                       |0.0696      |NULL

## 4. Prueba de lectura con un archivo válido

Primero se prueba un archivo conocido:

`yellow_tripdata_2023-01.parquet`

Esto confirma que Spark puede leer Parquet correctamente.

In [11]:
# Buscar el archivo yellow 2023-01.
# Esta prueba es solo para confirmar que Spark funciona antes de procesar todo.

test_file = None

for file_path in all_parquet_files:
    if file_path.name == "yellow_tripdata_2023-01.parquet":
        test_file = file_path
        break

if test_file is None:
    print("No se encontró yellow_tripdata_2023-01.parquet")
else:
    print("Archivo de prueba:", spark_path(test_file))

    df_test = spark.read.parquet(spark_path(test_file))

    print("Registros:", df_test.count())
    print("Columnas:", len(df_test.columns))

    df_test.show(5, truncate=False)

Archivo de prueba: C:/Users/Usuario/Downloads/etl_spark_parquet_advanced/etl_spark_parquet_advanced/data/raw/yellow/year=2023/month=01/yellow_tripdata_2023-01.parquet
Registros: 3066766
Columnas: 19
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-

In [12]:
# Revisar esquema del archivo de prueba.
# El esquema muestra columnas, tipos de datos y nulos.

if "df_test" in globals():
    df_test.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [13]:
# Mostrar columnas importantes si existen.
# Esto ayuda a entender rápidamente el contenido del archivo.

if "df_test" in globals():
    columns_to_show = [
        "VendorID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "fare_amount",
        "tip_amount",
        "total_amount"
    ]

    existing_columns = [c for c in columns_to_show if c in df_test.columns]

    df_test.select(existing_columns).show(10, truncate=False)

+--------+--------------------+---------------------+-------------+-----------+----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|tip_amount|total_amount|
+--------+--------------------+---------------------+-------------+-----------+----------+------------+
|2       |2023-01-01 00:32:10 |2023-01-01 00:40:36  |0.97         |9.3        |0.0       |14.3        |
|2       |2023-01-01 00:55:08 |2023-01-01 01:01:27  |1.1          |7.9        |4.0       |16.9        |
|2       |2023-01-01 00:25:04 |2023-01-01 00:37:49  |2.51         |14.9       |15.0      |34.9        |
|1       |2023-01-01 00:03:48 |2023-01-01 00:13:25  |1.9          |12.1       |0.0       |20.85       |
|2       |2023-01-01 00:10:29 |2023-01-01 00:21:19  |1.43         |11.4       |3.28      |19.68       |
|2       |2023-01-01 00:50:34 |2023-01-01 01:02:52  |1.84         |12.8       |10.0      |27.8        |
|2       |2023-01-01 00:09:22 |2023-01-01 00:19:49  |1.66       

## 5. Lectura de carpetas particionadas completas

En Windows, Spark puede fallar cuando se le pasa directamente una carpeta.  
Para evitar ese problema, aquí se leen los archivos internos de la carpeta uno por uno, pero juntos en un mismo DataFrame.

Ejemplo:

- Carpeta lógica: `data/raw/yellow/year=2023`
- Archivos internos: enero, febrero, marzo y abril de 2023

In [14]:
# Función para leer una carpeta particionada completa usando la lista de archivos.
# No modifica los archivos originales.

def read_partitioned_folder_by_files(folder_path):
    folder_path = Path(folder_path)
    parquet_files = sorted(folder_path.rglob("*.parquet"))

    if not parquet_files:
        raise FileNotFoundError(f"No se encontraron archivos Parquet en: {folder_path}")

    spark_paths = [spark_path(p) for p in parquet_files]

    df = spark.read.parquet(*spark_paths)

    return df, parquet_files


# Ejemplo: leer todos los yellow del año 2023
yellow_2023_folder = RAW_PATH / "yellow" / "year=2023"

try:
    df_yellow_2023, yellow_2023_files = read_partitioned_folder_by_files(yellow_2023_folder)

    print("Carpeta lógica leída:", yellow_2023_folder)
    print("Archivos leídos:")
    for p in yellow_2023_files:
        print("-", p.name)

    print("Registros totales yellow 2023:", df_yellow_2023.count())
    print("Columnas:", len(df_yellow_2023.columns))

except Exception as e:
    print("No se pudo leer la carpeta particionada.")
    print(clean_error_message(e))

Carpeta lógica leída: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw\yellow\year=2023
Archivos leídos:
- yellow_tripdata_2023-01.parquet
- yellow_tripdata_2023-02.parquet
- yellow_tripdata_2023-03.parquet
- yellow_tripdata_2023-04.parquet
Registros totales yellow 2023: 12672737
Columnas: 19


In [15]:
# Lectura de carpetas particionadas por servicio.
# Esto demuestra que Spark puede leer carpetas completas por servicio.

partitioned_folders_to_test = [
    RAW_PATH / "yellow" / "year=2023",
    RAW_PATH / "green" / "year=2023",
    RAW_PATH / "fhvhv" / "year=2023"
]

for folder in partitioned_folders_to_test:
    try:
        df_folder, files_read = read_partitioned_folder_by_files(folder)

        print("=" * 80)
        print("Carpeta leída:", folder)
        print("Archivos encontrados:", len(files_read))
        print("Registros:", df_folder.count())
        print("Columnas:", len(df_folder.columns))

    except Exception as error:
        print("=" * 80)
        print("No se pudo leer la carpeta:", folder)
        print("Error:", str(error)[:300])

Carpeta leída: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw\yellow\year=2023
Archivos encontrados: 4
Registros: 12672737
Columnas: 19
Carpeta leída: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw\green\year=2023
Archivos encontrados: 2
Registros: 133020
Columnas: 20
Carpeta leída: c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\raw\fhvhv\year=2023
Archivos encontrados: 1
Registros: 18479031
Columnas: 24


## 6. Crear `audit_file_inventory`

Ahora se procesa cada archivo de forma individual.

Por cada archivo se registra:

- `process_id`
- `source_system`
- `service_type`
- `file_name`
- `file_path`
- `file_size_mb`
- `partition_year`
- `partition_month`
- `read_status`
- `record_count`
- `column_count`
- `schema_hash`
- `error_message`
- `processed_at`

La ventaja de leer archivo por archivo es que **si uno falla, el pipeline no se detiene**.

In [16]:
# Esquema obligatorio de la tabla audit_file_inventory.

audit_schema = StructType([
    StructField("process_id", StringType(), False),
    StructField("source_system", StringType(), True),
    StructField("service_type", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("file_size_mb", DoubleType(), True),
    StructField("partition_year", IntegerType(), True),
    StructField("partition_month", IntegerType(), True),
    StructField("read_status", StringType(), True),
    StructField("record_count", LongType(), True),
    StructField("column_count", IntegerType(), True),
    StructField("schema_hash", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("processed_at", TimestampType(), True)
])

In [17]:
# Procesamiento individual de archivos.
# Si un archivo está corrupto, se captura el error y se continúa con el siguiente.

inventory_rows = []

for index, file_path in enumerate(all_parquet_files, start=1):
    service_type = detect_service_type(file_path)
    source_system = detect_source_system(service_type)
    processed_at = datetime.now()

    print(f"Procesando {index}/{len(all_parquet_files)}: {file_path.name}")

    try:
        if Path(file_path).stat().st_size == 0:
            raise ValueError("Archivo vacío: tamaño 0 bytes")

        df_tmp = spark.read.parquet(spark_path(file_path))

        record_count = df_tmp.count()
        column_count = len(df_tmp.columns)
        hash_value = schema_hash(df_tmp.schema)

        read_status = "READ_OK"
        error_message = None

    except Exception as e:
        record_count = None
        column_count = None
        hash_value = None
        read_status = "READ_ERROR"
        error_message = clean_error_message(e)

    inventory_rows.append({
        "process_id": PROCESS_ID,
        "source_system": source_system,
        "service_type": service_type,
        "file_name": file_path.name,
        "file_path": spark_path(file_path),
        "file_size_mb": file_size_mb(file_path),
        "partition_year": extract_partition_value(file_path, "year"),
        "partition_month": extract_partition_value(file_path, "month"),
        "read_status": read_status,
        "record_count": record_count,
        "column_count": column_count,
        "schema_hash": hash_value,
        "error_message": error_message,
        "processed_at": processed_at
    })

print("Inventario generado correctamente")

Procesando 1/16: ARROW-GH-41317.parquet
Procesando 2/16: ARROW-GH-41321.parquet
Procesando 3/16: ARROW-GH-45185.parquet
Procesando 4/16: ARROW-RS-GH-6229-DICTHEADER.parquet
Procesando 5/16: PARQUET-1481.parquet
Procesando 6/16: fhvhv_tripdata_2023-01.parquet
Procesando 7/16: green_tripdata_2023-01.parquet
Procesando 8/16: green_tripdata_2023-02.parquet
Procesando 9/16: yellow_tripdata_2020-01.parquet
Procesando 10/16: yellow_tripdata_2021-01.parquet
Procesando 11/16: yellow_tripdata_2022-01.parquet
Procesando 12/16: yellow_tripdata_2022-02.parquet
Procesando 13/16: yellow_tripdata_2023-01.parquet
Procesando 14/16: yellow_tripdata_2023-02.parquet
Procesando 15/16: yellow_tripdata_2023-03.parquet
Procesando 16/16: yellow_tripdata_2023-04.parquet
Inventario generado correctamente


In [18]:
# Crear DataFrame Spark con el inventario.
# Esta es la tabla principal de la Fase 1.

audit_file_inventory = spark.createDataFrame(inventory_rows, schema=audit_schema)

audit_file_inventory.orderBy(
    "service_type",
    "partition_year",
    "partition_month",
    "file_name"
).show(100, truncate=False)

+------------------------------------+----------------------+------------+-----------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------+---------------+-----------+------------+------------+--------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [19]:
# Resumen del inventario.
# Sirve para verificar rápidamente cuántos archivos se leyeron bien y cuántos fallaron.

audit_file_inventory.groupBy(
    "source_system",
    "service_type",
    "read_status"
).agg(
    F.count("*").alias("total_files"),
    F.sum(F.coalesce(F.col("record_count"), F.lit(0))).alias("total_records")
).orderBy(
    "source_system",
    "service_type",
    "read_status"
).show(truncate=False)

+----------------------+------------+-----------+-----------+-------------+
|source_system         |service_type|read_status|total_files|total_records|
+----------------------+------------+-----------+-----------+-------------+
|APACHE_PARQUET_TESTING|bad_parquet |READ_ERROR |3          |0            |
|APACHE_PARQUET_TESTING|bad_parquet |READ_OK    |2          |5            |
|NYC_TLC               |fhvhv       |READ_OK    |1          |18479031     |
|NYC_TLC               |green       |READ_OK    |2          |133020       |
|NYC_TLC               |yellow      |READ_OK    |8          |25890876     |
+----------------------+------------+-----------+-----------+-------------+



## 7. Guardar `audit_file_inventory` en la capa audit

Se guarda la tabla de auditoría.

In [20]:
# Guardar audit_file_inventory en formato Parquet
# Este es el formato correcto para mantener la arquitectura Lakehouse.

audit_parquet_path = AUDIT_PATH / f"audit_file_inventory_{PROCESS_ID}"

audit_file_inventory.write.mode("overwrite").parquet(
    str(audit_parquet_path)
)

print("audit_file_inventory guardado correctamente en formato Parquet:")
print(audit_parquet_path)

audit_file_inventory guardado correctamente en formato Parquet:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\audit\audit_file_inventory_2e9a7366-d209-4985-ad78-8c8ca2a14476


## 8. Crear evidencia de cuarentena para archivos con error

No se ignoran archivos dañados.  
Si un archivo falla, queda registrado en `data/quarantine`.

En esta fase se guarda una **referencia al archivo**, no se borra ni se modifica el original.

In [ ]:














































# Crear tabla de archivos enviados a cuarentena.

quarantine_schema = StructType([
    StructField("process_id", StringType(), False),
    StructField("source_system", StringType(), True),
    StructField("service_type", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("rejection_stage", StringType(), True),
    StructField("error_category", StringType(), True),
    StructField("technical_reason", StringType(), True),
    StructField("recommended_action", StringType(), True),
    StructField("rejected_at", TimestampType(), True)
])

quarantine_rows = []

failed_files = audit_file_inventory.filter(F.col("read_status") != "READ_OK").collect()

for row in failed_files:
    error_category = classify_file_error(row["file_path"], row["error_message"])

    quarantine_rows.append({
        "process_id": row["process_id"],
        "source_system": row["source_system"],
        "service_type": row["service_type"],
        "file_name": row["file_name"],
        "file_path": row["file_path"],
        "rejection_stage": "EXTRACTION",
        "error_category": error_category,
        "technical_reason": row["error_message"],
        "recommended_action": "Revisar archivo original, validar formato Parquet y decidir si se reemplaza o se excluye.",
        "rejected_at": datetime.now()
    })

quarantine_files_df = spark.createDataFrame(quarantine_rows, schema=quarantine_schema)

quarantine_files_df.show(100, truncate=False)

+------------------------------------+----------------------+------------+----------------------+----------------------------------------------------------------------------------------------------------------------------+---------------+---------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [22]:
# Guardar referencias de archivos en cuarentena en formato Parquet

quarantine_parquet_path = QUARANTINE_PATH / f"quarantine_file_references_{PROCESS_ID}"

quarantine_files_df.write.mode("overwrite").parquet(
    str(quarantine_parquet_path)
)

print("Referencias de cuarentena guardadas correctamente en formato Parquet:")
print(quarantine_parquet_path)

Referencias de cuarentena guardadas correctamente en formato Parquet:
c:\Users\Usuario\Downloads\etl_spark_parquet_advanced\etl_spark_parquet_advanced\data\quarantine\quarantine_file_references_2e9a7366-d209-4985-ad78-8c8ca2a14476


## 9. Validación final de la Fase 1

Aquí comprobamos que se cumple lo solicitado:

- Existen archivos Parquet en `raw`.
- Se leyó cada archivo individualmente.
- Se creó `audit_file_inventory`.
- Se registraron errores de lectura.
- Los archivos problemáticos quedaron referenciados en `quarantine`.

In [23]:
# Validaciones automáticas de aceptación.

required_columns = [
    "process_id",
    "source_system",
    "service_type",
    "file_name",
    "file_path",
    "file_size_mb",
    "partition_year",
    "partition_month",
    "read_status",
    "record_count",
    "column_count",
    "schema_hash",
    "error_message",
    "processed_at"
]

inventory_columns = audit_file_inventory.columns

missing_inventory_columns = [c for c in required_columns if c not in inventory_columns]

total_files = audit_file_inventory.count()
read_ok_files = audit_file_inventory.filter(F.col("read_status") == "READ_OK").count()
read_error_files = audit_file_inventory.filter(F.col("read_status") != "READ_OK").count()
quarantine_files = quarantine_files_df.count()
distinct_schemas = audit_file_inventory.filter(
    F.col("schema_hash").isNotNull()
).select("schema_hash").distinct().count()

print("VALIDACIÓN FASE 1")
print("Total de archivos inventariados:", total_files)
print("Archivos leídos correctamente:", read_ok_files)
print("Archivos con error de lectura:", read_error_files)
print("Archivos registrados en cuarentena:", quarantine_files)
print("Esquemas distintos detectados en archivos READ_OK:", distinct_schemas)

print("\nTOTALES POR SERVICE_TYPE")
audit_file_inventory.groupBy("service_type").agg(
    F.count("*").alias("total_files"),
    F.sum(F.when(F.col("read_status") == "READ_OK", 1).otherwise(0)).alias("read_ok"),
    F.sum(F.when(F.col("read_status") != "READ_OK", 1).otherwise(0)).alias("read_error"),
    F.countDistinct("schema_hash").alias("distinct_schemas")
).orderBy("service_type").show(truncate=False)

print("TOTALES POR FUENTE")
audit_file_inventory.groupBy("source_system").agg(
    F.count("*").alias("total_files")
).orderBy("source_system").show(truncate=False)

if missing_inventory_columns:
    print("FALTAN COLUMNAS EN audit_file_inventory:")
    for col in missing_inventory_columns:
        print("-", col)
else:
    print("La tabla audit_file_inventory tiene todas las columnas requeridas.")

if total_files > 0 and read_ok_files > 0:
    print("Criterio principal cumplido: existen archivos válidos leídos con Spark.")
else:
    print("Revisar: no se detectaron archivos válidos.")

if read_error_files == quarantine_files:
    print("Criterio de cuarentena cumplido: todo error quedó registrado.")
else:
    print("Revisar: la cantidad de errores no coincide con cuarentena.")

print("\nCONCLUSIÓN INTERPRETATIVA")
print(
    "La Fase 1 demuestra que las fuentes no son homogéneas: existen errores físicos/lógicos "
    "y múltiples esquemas válidos. La Fase 2 deberá reconstruir esquemas canónicos por servicio, "
    "mientras las fases posteriores deberán conservar trazabilidad y tratamiento diferenciado de cuarentena."
)

VALIDACIÓN FASE 1
Total de archivos inventariados: 16
Archivos leídos correctamente: 13
Archivos con error de lectura: 3
Archivos registrados en cuarentena: 3
Esquemas distintos detectados en archivos READ_OK: 8

TOTALES POR SERVICE_TYPE
+------------+-----------+-------+----------+----------------+
|service_type|total_files|read_ok|read_error|distinct_schemas|
+------------+-----------+-------+----------+----------------+
|bad_parquet |5          |2      |3         |2               |
|fhvhv       |1          |1      |0         |1               |
|green       |2          |2      |0         |2               |
|yellow      |8          |8      |0         |3               |
+------------+-----------+-------+----------+----------------+

TOTALES POR FUENTE
+----------------------+-----------+
|source_system         |total_files|
+----------------------+-----------+
|APACHE_PARQUET_TESTING|5          |
|NYC_TLC               |11         |
+----------------------+-----------+

La tabla audit_

## 10. Resultado

Al finalizar este notebook se obtuvo dos salidas principales:

### 1. `audit_file_inventory`

Tabla técnica que dice:

- Qué archivos se encontraron.
- Cuáles se pudieron leer.
- Cuántos registros tienen.
- Cuántas columnas tienen.
- Qué esquema tienen.
- Qué error ocurrió si fallaron.

### 2. `quarantine_file_references`

Tabla de control que dice:

- Qué archivos fallaron.
- Por qué fallaron.
- En qué etapa fallaron.
- Qué acción se recomienda.

Con esto ya se demuestra que el pipeline **no se cae por un archivo corrupto** y que los errores quedan documentados.

In [24]:
# Vista final rápida del inventario.

audit_file_inventory.select(
    "process_id",
    "service_type",
    "file_name",
    "partition_year",
    "partition_month",
    "read_status",
    "record_count",
    "column_count",
    "schema_hash",
    "error_message"
).orderBy(
    "service_type",
    "partition_year",
    "partition_month",
    "file_name"
).show(100, truncate=False)

+------------------------------------+------------+-----------------------------------+--------------+---------------+-----------+------------+------------+--------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [25]:
# Cerrar Spark solo si ya terminaste de trabajar.
# Si vas a seguir probando celdas, no ejecutes esta celda.

# spark.stop()